# Этап 2: Qwen/Qwen3.5-4B (`q35_4b_v32_m1c300`)

Дообучение `q35_4b_v32_m1` с `checkpoint-300`: 16.11.2026 обучение прошло все 500 шагов, но на Drive не попали веса после шага 350 и итоговый адаптер; ответы 17.11 дала промежуточная точка 350. Папка на Drive та же (`mcnl/q35_4b_v32_m1`), настройки те же, осталось 200 шагов (около часа). Номера ячеек - как в `stage2_q35_4b_v32_m1`; пробы (4, 5) и быстрой проверки (9) здесь нет.

QLoRA: веса модели заморожены в 4 битах, обучается только добавочный слой LoRA. Среда выполнения - GPU (T4). Результаты пишутся на Drive в `mcnl/<RUN>`; после обрыва сессии ячейки запускаются заново, обучение и ответы продолжаются с места обрыва.

## 1. Среда

In [ ]:
%cd /content
!pip -q install -U transformers peft accelerate bitsandbytes
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Drive и код проекта
Код берётся из GitHub. Для закрытого репозитория - секрет `GITHUB_TOKEN` (только чтение) в панели ключей слева.

In [ ]:
import os, subprocess
os.chdir('/content')
from google.colab import drive, userdata
drive.mount('/content/drive')
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
REPO = 'github.com/xdstk/lang.cir.spice.mc12'
!rm -rf /content/repo
done = subprocess.run(['git', 'clone', '-q', '--depth', '1', f'https://x-access-token:{token}@{REPO}' if token else f'https://{REPO}', '/content/repo'],
                      capture_output=True, text=True, env={'GIT_TERMINAL_PROMPT': '0', 'PATH': '/usr/bin:/bin:/usr/local/bin'})
if done.returncode:
    raise SystemExit('Код не скачан: ' + done.stderr.replace(token or '<нет>', '***').strip())
!git -C /content/repo remote set-url origin https://{REPO}
%cd /content/repo
!git log --oneline -1

## 3. Настройки

In [ ]:
MODEL = 'Qwen/Qwen3.5-4B'
RUN = 'q35_4b_v32_m1'  # папка на Drive
DATA = 'data/dataset/v3_2/mode1'  # датасет обучения
INSTR = 'v3'           # инструкция модели при ответах: как при обучении
SUBSET = 2000          # подмножество train: 1000, 2000 или 3000
LIMIT = 0              # 0 - всё подмножество
FORMAT = 'netlist'     # netlist или ir
EPOCHS = 2
BATCH = 2              # шаг оптимизации всегда 8 записей
ACCUM = 8 // BATCH
SEED = 1
OUT = f'/content/drive/MyDrive/mcnl/{RUN}'
!mkdir -p {OUT}
!python src/mcnl/training.py --data {DATA} --subset {SUBSET} --limit {LIMIT} --format {FORMAT} --out out/training/{RUN}

## 3а. Подготовка к продолжению
Один раз перед ячейкой 6; после обрыва сессии её можно запускать снова - готовый адаптер и новые ответы она не трогает. Ответы точки 350 переносятся в `ckpt350/`. Неполные контрольные точки (350 без `optimizer.pt`, пустые 400-500) удалит сама ячейка 6: с ними Trainer удалял бы точку продолжения и новые точки.

In [ ]:
import glob, os, shutil, sys
sys.path.insert(0, 'src')
from mcnl import hf_runtime
if os.path.exists(f'{OUT}/adapter/adapter_model.safetensors'):
    print('адаптер с весами есть: обучение закончено, ячейку 6 можно пропустить')
else:
    if os.path.exists(f'{OUT}/run.json'):  # run.json без весов адаптера - след прерванной выгрузки
        os.makedirs(f'{OUT}/ckpt350', exist_ok=True)
        old = glob.glob(f'{OUT}/*.jsonl') + glob.glob(f'{OUT}/loss.png')
        for f in old:
            shutil.move(f, f'{OUT}/ckpt350/' + os.path.basename(f))
        os.remove(f'{OUT}/run.json')
        shutil.rmtree(f'{OUT}/adapter', ignore_errors=True)
        print(f'ответы точки 350 ({len(old)} файлов) -> ckpt350/; run.json и adapter/ без весов удалены')
    else:
        print('подготовка уже сделана')
    print('ячейка 6 продолжит с', hf_runtime.last_complete(f'{OUT}/checkpoints') or 'нуля: полных контрольных точек нет (около 2,4 ч)')

## 6. Обучение
`loss` должен падать; `nan` или плато - ошибка.

В начале вывода - «неполная контрольная точка удалена: …» и «продолжаю с …/checkpoint-300».

In [ ]:
!python colab/train_lora.py --model {MODEL} --data out/training/{RUN} --out {OUT} --epochs {EPOCHS} --seed {SEED} --batch {BATCH} --accum {ACCUM}

## 7. График обучения
Сохраняется на Drive как `loss.png`.

In [ ]:
import json, matplotlib.pyplot as plt
run = json.load(open(f'{OUT}/run.json', encoding='utf-8'))
tr = [(x['step'], x['loss']) for x in run['log'] if 'loss' in x]
ev = [(x['step'], x['eval_loss']) for x in run['log'] if 'eval_loss' in x]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label='обучающая выборка')
if ev:
    plt.plot(*zip(*ev), 'o-', label='dev')
plt.xlabel('шаг обучения'); plt.ylabel('потери'); plt.legend(); plt.grid(alpha=0.3)
plt.title(RUN); plt.savefig(f'{OUT}/loss.png', dpi=200, bbox_inches='tight'); plt.show()
print(f"Время обучения: {run['train_seconds'] / 60:.1f} мин (все сессии), пик памяти GPU: {run['gpu_peak_gb']:.1f} ГБ, GPU: {run['gpu']}")
print('Отброшено длинных примеров (train, dev):', run['dropped_long'])
print('Версии:', run['versions'])

## 8. Ответы обученной модели
30 записей train, весь dev (150), запросы C01.

In [ ]:
A = f'--model {MODEL} --adapter {OUT}/adapter --format {FORMAT} --name {RUN} --instructions {INSTR}'
!python colab/predict.py {A} --records {DATA}/train.jsonl --limit 30 --out {OUT}/train30.jsonl
!python colab/predict.py {A} --records {DATA}/dev.jsonl --out {OUT}/dev.jsonl
!python colab/predict.py {A} --records data/human_test/v1/paraphrase.jsonl --out {OUT}/paraphrase.jsonl

### 8б. Выборки для наблюдения
Из test v3 и test v2 - все запросы Саллен-Ки и по 70 случайных (около 200), затем llm_test и llm_test2 целиком (около 240) и llm_test3, если она уже есть в репозитории. Всего около 30 мин.

In [ ]:
import os
A = f'--model {MODEL} --adapter {OUT}/adapter --format {FORMAT} --name {RUN} --instructions {INSTR}'
for s in ['watch_v3', 'watch_v2']:
    !python colab/predict.py {A} --records {DATA}/{s}.jsonl --out {OUT}/{s}.jsonl
LLM = [('llm_test', 'data/dataset/llm_test_v1/llm_test.jsonl'), ('llm_test_chatty', 'data/dataset/llm_test_v1/llm_test_chatty.jsonl'),
       ('llm_test2', 'data/dataset/llm_test2_v1/llm_test2.jsonl'), ('llm_test2_chatty', 'data/dataset/llm_test2_v1/llm_test2_chatty.jsonl'),
       ('llm_test3', 'data/dataset/llm_test3_v1/llm_test3.jsonl')]
for s, path in LLM:
    if os.path.exists(path):
        !python colab/predict.py {A} --records {path} --out {OUT}/{s}.jsonl

### 8в. Запросы программы, режим 1
Справочник, меню «Примеры», вкладка «Правила» - около 40 запросов, несколько минут. Проверка - локально: `python src/mcnl/program_prompts.py --check program.jsonl`, нужно 100%.

In [ ]:
import os
A = f'--model {MODEL} --adapter {OUT}/adapter --format {FORMAT} --name {RUN} --instructions {INSTR}'
if os.path.exists('data/program_prompts/program.jsonl'):
    !python colab/predict.py {A} --records data/program_prompts/program.jsonl --mode 1 --out {OUT}/program.jsonl